# 01 — 라운드를 거치며 여정이 만들어지는 과정을 살펴보기

[English](01_rounds_labels_and_journeys.ipynb) | **한국어**

기차 안이 아닌 역 밖에서 시작해 봅시다. 가상의 출발지 `O`에서는 플랫폼 `A`까지 ENTRY 도보 이동이 필요합니다. 목적지 `Z`에는 `D`에서 나가는 EXIT 도보 이동이 포함됩니다.

모든 탑승과 모든 시각 값을 설명하는 것이 목표입니다. 이 정류장과 시각은 인위적으로 만든 것이며, 이 노트북은 실제 여정 추천이 아닙니다. [01장](../docs/01_rounds_labels_and_pareto.ko.md)과 함께 읽어 보세요.

In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
if not (root / "src").is_dir() and (root.parent / "src").is_dir():
    root = root.parent
if not (root / "src" / "raptor.py").is_file():
    raise RuntimeError("Start this notebook from the repository root or notebooks directory.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

In [ ]:
from src import compile_timetable, demo_timetable, raptor, WalkPolicy, parse_time, format_time
index = compile_timetable(demo_timetable())
ready = parse_time("08:00:00")
result = raptor(index, "O", "Z", ready, max_boardings=3, boarding_slack=60)
for k, row in enumerate(result.rows):
    print(f"At most {k} boardings:", {stop: format_time(label.time) for stop, label in sorted(row.items())})

## 먼저 확인하기: 라운드는 무엇을 세는가?

라운드 0에서는 도보 이동이 가능합니다. 라운드 1에서는 한 대의 차량을 이용할 수 있습니다. 라운드 2에서는 두 대의 차량을 이용할 수 있는데, 이는 두 번이 아니라 **한 번의 환승**을 뜻합니다.

목적지는 라운드 0에서는 없어야 하고, 라운드 1에서는 08:30에 도달 가능하며, 라운드 2에서는 08:22로 개선되어야 합니다. `D`의 08:21 라벨은 목적지 도착이 아닙니다. EXIT에 1분이 더 걸립니다.

In [ ]:
assert "Z" not in result.rows[0]
assert result.rows[1]["Z"].time == parse_time("08:30:00")
assert result.rows[2]["Z"].time == parse_time("08:22:00")
assert result.rows[2]["D"].time == parse_time("08:21:00")
print("Round invariant checks passed.")

## 유용한 두 선택지를 재구성하기

빠른 선택지는 두 번 탑승합니다. 느린 선택지는 환승을 피합니다. 도착 시각과 탑승 횟수 모두에서 어느 한쪽도 다른 쪽을 지배하지 않습니다.

구간 사이의 대기는 암묵적으로 포함됩니다. `Journey.validate`는 연속성, 시간 순서, 탑승 여유 시간, 최종 종점, 탑승 횟수를 확인합니다.

In [ ]:
for journey in result.journeys():
    journey.validate(boarding_slack=60)
    print(f"\nArrive {format_time(journey.arrival)} / {journey.transfers} transfers / {journey.walking_seconds}s walking")
    for leg in journey.legs:
        print(leg.kind, leg.source, "->", leg.target, format_time(leg.departure), format_time(leg.arrival), leg.trip_id or "")

## 준비 시각을 1초 옮기기

진입 시간과 탑승 여유 시간을 합치면 첫 탑승 임계값은 출발지 준비 시각 정확히 08:00:00이 됩니다. 같아도 허용됩니다. 1초 뒤는 같은 질의가 아닙니다.

In [ ]:
late = raptor(index, "O", "Z", ready + 1, max_boardings=3, boarding_slack=60)
assert result.journeys()[0].arrival == parse_time("08:22:00")
assert late.journeys()[0].arrival == parse_time("08:27:00")
print("At 08:00:00:", format_time(result.journeys()[0].arrival))
print("At 08:00:01:", format_time(late.journeys()[0].arrival))

## 순위 가중치가 아니라 강한 제약을 바꾸기

이제 계단을 명시적으로 허용합니다. 이는 새 요청에서 가능한 도보 그래프를 바꿉니다. 엄격 모드가 실패한 뒤 자동으로 복구하는 경로가 아닙니다.

In [ ]:
stairs_allowed = raptor(index, "O", "Z", ready, max_boardings=3, boarding_slack=60, policy=WalkPolicy(False))
assert stairs_allowed.journeys()[0].arrival == parse_time("08:21:00")
print("Strict step-free:", format_time(result.journeys()[0].arrival))
print("Stairs explicitly allowed:", format_time(stairs_allowed.journeys()[0].arrival))
print("Measured operations:", result.metrics)

## 로컬 계산 추적하기

`src/raptor.py`에서 `affected_routes`, `scan_route`, `close_footpaths`를 따라가 보세요. 04장은 이 함수들을 테스트 데이터 및 경로 증거 검사와 연결합니다. 이 노트북은 제한된 로컬 모델을 보여 줍니다. 여러 운행일의 입력을 불러오는 기능과 완전한 다기준 상태는 아직 구현되지 않았습니다.

계속하기 전에 탑승 중 `previous` 대신 `current`를 읽으면 왜 차량 하나가 더 필요한 경우를 숨길 수 있는지 설명해 보세요. 그다음 `python -m pytest -q`를 실행하고 경로 순서 회귀 테스트를 찾아보세요.